# 4. Weight ports — on-chip, classic streaming, and DDR

By default, layers store weights on chip and initialize them from MIF files
(`mif_init=True`).

When the model needs more memory than available on the FPGA, offload weights to
**DDR** over an **AXI memory-mapped** interface. Layers that support DDR
offload today:

- `Dot`
- `Split_dot`
- `Add_bias`
- `Multiply_bias`
- `Kv_cache`

Any leaf that connects to a DAG weight port must use `mif_init=False`.


In [1]:
from pathlib import Path

import numpy as np
from fpga_ai_suite.spatial.flow import run_flow
from fpga_ai_suite.spatial.layers import (
    Add_bias,
    Directed_acyclic_graph,
    Dot,
    Split_dot,
)
from fpga_ai_suite.spatial.compiler.composite_layers.kv_cache import Kv_cache

WORK = Path("work_weight_ports")
WORK.mkdir(exist_ok=True)


## Creating weight ports

On a DAG / advanced layer:

```python
self.create_weight_port(
    name,
    internal_rd_channel_width=...,
    internal_wr_channel_width=None,  # needed for writers such as Kv_cache
    ddr_addr_width=None,             # AXI interface
    ddr_channel_width=None,          # AXI interface
    bank=0,
    start_addr=0,                    # bytes; multiple of ddr_channel_width/8
    capacity=4,                      # GiB
)
```

Omit `ddr_addr_width` / `ddr_channel_width` for **classic** on-chip streaming
ports. Set both to enable **DDR**.

A layer connected to DDR **MUST** have at least three "pages". For bias layers, a page is a vector meaning its weight dimension must be 2. For dots and kv_cache, a page is a matrix so its weights must have three dimensions.

### Channel width rules

- Each leaf exposes `get_max_channel_width_for_weight_port(...)`. This is tied to num_lanes/num_lanes_out
- Most layers accept an internal width **≤** that max.
- **Kv_cache** requires the internal width to be **exactly** the max.
- `ddr_channel_width` must be an **integer multiple** of `internal_rd_channel_width`.


In [2]:
# Inspect max channel width on a standalone Dot (before wrapping)
probe = Dot(
    weights=np.random.randn(8, 8).astype(np.float32),
    num_lanes_in=4,
    num_lanes_out=4,
    dtype="BF16",
    mif_init=False,
)
print("Dot max weight channel width:", probe.get_max_channel_width_for_weight_port(probe.weights["weights"]))


Dot max weight channel width: 128


## Manual connect vs `auto_connect_weights`

- **Manual:** `self.weights["weights0"].connect(leaf.weights["weights"])`
- **`Split_dot` / `Kv_cache`:** can use several channels. If you create
  `weights0` … `weights{N-1}` with the **same count** as `num_channels`,
  `auto_connect_weights()` pairs them by name.
- Layers with a **single** weight port are auto-connected in **round-robin**
  across root ports when you call `auto_connect_weights()`.

Always set `mif_init=False` on leaves that should attach to these ports.


In [3]:
class Ddr_split_dot_dag(Directed_acyclic_graph):
    """Tiny Split_dot example with DDR weight ports and autoconnect.

    For channel_split_mode='input', num_split_input must be >= num_channels and
    evenly divisible by it (e.g. num_split_input=2, num_channels=2).
    """

    def __init__(self, weights, bias, num_split=4, num_channels=2, num_lanes_in=4, num_lanes_out=4):
        self.weights_data = weights
        self.bias_data = bias
        self.num_split = num_split
        self.num_channels = num_channels
        self.num_lanes_in = num_lanes_in
        self.num_lanes_out = num_lanes_out
        super().__init__(module_name="Ddr_split_dot_dag")

    def init_topology(self):
        self.input = self.create_input("din", 16, self.num_lanes_in, "BF16")
        self.output = self.create_output("dout", 16, self.num_lanes_out, "BF16")

        # Bias is the bottleneck here
        internal_w = 16 * self.num_lanes_out
        for i in range(self.num_channels):
            self.create_weight_port(
                f"weights{i}",
                internal_rd_channel_width=internal_w,
                ddr_addr_width=32,
                ddr_channel_width=512,
                bank=i,
                capacity=1,
            )

        # Split dot reduces on chip memory usage by tiling the weights on external memory.
        # A split of 4, with 2 channels will in total use 2x less on chip memory, since
        # each channel requires a dedicated dot core consuming 2x more resources
        self.split_dot = self.add_node(
            Split_dot(
                weights=self.weights_data,
                num_split_input=1,
                num_split_output=self.num_split,
                num_channels=self.num_channels,
                channel_split_mode="output",
                num_lanes_in=self.num_lanes_in,
                num_lanes_out=self.num_lanes_out,
                dtype="BF16",
                mif_init=False,
            )
        )

        self.add_bias = self.add_node(
            Add_bias(
                bias=self.bias_data,
                num_lanes=self.num_lanes_out,
                dtype="BF16",
                mif_init=False,
            )
        )
            
        self.auto_connect_weights()

    def forward(self, din):
        x = self.split_dot(din)
        x = self.add_bias(x)
        return x,


# features_in must be divisible by num_split_input (16 % 2 == 0)
w = np.random.randn(3, 64, 128).astype(np.float32)
b = np.random.randn(3, 128).astype(np.float32)
ddr_model = Ddr_split_dot_dag(w, b)
print("weight ports:", list(ddr_model.weights.keys()))
print("Weights0 connections:", [dest.owner.module_name for dest in ddr_model.weights["weights0"].dests])
print("Weights1 connections:", [dest.owner.module_name for dest in ddr_model.weights["weights1"].dests])


weight ports: ['weights0', 'weights1']
Weights0 connections: ['split_dot', 'add_bias_bf16']
Weights1 connections: ['split_dot']


In [4]:
class Manual_dot_ddr_dag(Directed_acyclic_graph):
    """Single Dot with one DDR weight port, connected manually."""

    def __init__(self, weights, num_lanes=4):
        self.weights_data = weights
        self.num_lanes = num_lanes
        super().__init__(module_name="Manual_dot_ddr_dag")

    def init_topology(self):
        self.input = self.create_input("din", 16, self.num_lanes, "BF16")
        self.output = self.create_output("dout", 16, self.num_lanes, "BF16")

        self.dot = self.add_node(
            Dot(
                weights=self.weights_data,
                num_lanes_in=self.num_lanes,
                num_lanes_out=self.num_lanes,
                dtype="BF16",
                mif_init=False,
            )
        )
        max_w = self.dot.get_max_channel_width_for_weight_port(self.dot.weights["weights"])
        self.create_weight_port(
            "weights0",
            internal_rd_channel_width=max_w,
            ddr_addr_width=32,
            ddr_channel_width=512,
            bank=0,
            capacity=1,
        )
        # Manual connect by name / port object
        # Layers that only have one weight port will use "weights" as the port name
        # whereas layers with multiple weight ports will use "weights{i}"
        self.weights["weights0"].connect(self.dot.weights["weights"])

        self.connect(self.input, self.dot.inputs.din)
        self.connect(self.dot.outputs.dout, self.output)


manual = Manual_dot_ddr_dag(np.random.randn(3, 64, 128).astype(np.float32))
print("Weights0 connections:", [dest.owner.module_name for dest in manual.weights["weights0"].dests])


Weights0 connections: ['dot_bf16']


## Estimates after DDR connections — `prepare_for_dumping`

Weight-port metadata (channel width/interface type)
is finalized by calling any of the dumper functions.

Call any of them before `estimate_*`. 

To change configurations afterward, **reinstantiate**
the model. Metadata is not meant to be updated.


In [5]:
est_model = Ddr_split_dot_dag(w, b)
est_model.set_shape(din_shape=(2, 1, 64))

dump_dir = WORK / "rtl"
est_model.dump_rtl("my_rtl_file.sv", dump_path=dump_dir)

num_lanes_in = est_model.inputs.din.num_lanes
input_packet_size = np.prod(est_model.inputs.din.shape[-2:]) // num_lanes_in
# Two packets back to back
input_timing = [(0, input_packet_size - 1), (input_packet_size, input_packet_size*2 - 1)]
est_model.inputs.din.data_timing = input_timing
output_timing, = est_model.estimate_timing()

print("latency (cycles):", output_timing[0][0] - input_timing[0][0])
print("resources:", est_model.estimate_resource_utilization())


latency (cycles): 681
resources: {'ALM': 2368, 'DSP': 30, 'M20K': 26, 'M80K': 0}


## Dump formats

After `forward` + `run_flow` (RTL dump), weight payloads land under `dump_files/`:

| Mode | File |
|------|------|
| **DDR AXI MM** | `weights_ddr_{bank}_{start_address_in_bytes}.bin` |
| **Classic streaming** (`mif_init=False`, no DDR args) | `{weight_port_name}.txt` — ASCII bitstrings, one beat per line |
| **On-chip MIF** (`mif_init=True`) | `init_file_*.mif` |


In [6]:
dump_model = Manual_dot_ddr_dag(np.random.randn(3, 64, 128).astype(np.float32))
data = np.random.randn(2, 1, 64).astype(np.float32)
dump_model(din=data)

dump_dir = WORK / "ddr_dump"
ret = run_flow(dump_model, output_path=dump_dir)
print("weight artifacts saved at: ", dump_dir)


Dumping input and output data...
Dumping weights: matrix size: [64,128] total weights: 12288
Running model MANUAL_DOT_DDR_DAG
weight artifacts saved at:  work_weight_ports/ddr_dump


## Classic streaming weight ports

Classic weight ports (no DDR args) expose **runtime-reconfigurable** streaming
weights. They do **not** save M20Ks: weights still live on chip. Any
weight-storing layer can connect if `mif_init=False`.

Each line in `{weight_port_name}.txt` corresponds to one beat during the channel streaming.


In [7]:
class Classic_weight_dag(Directed_acyclic_graph):
    def __init__(self, weights, bias, num_lanes=4):
        self.weights_data = weights
        self.bias_data = bias
        self.num_lanes = num_lanes
        super().__init__(module_name="Classic_weight_dag")

    def init_topology(self):
        self.input = self.create_input("din", 16, self.num_lanes, "BF16")
        self.output = self.create_output("dout", 16, self.num_lanes, "BF16")
        # Classic streaming port — no ddr_* arguments
        self.create_weight_port("weights", internal_rd_channel_width=64)

        self.dot = self.add_node(
            Dot(
                weights=self.weights_data,
                num_lanes_in=self.num_lanes,
                num_lanes_out=self.num_lanes,
                dtype="BF16",
                mif_init=False,
            )
        )
        self.bias = self.add_node(
            Add_bias(
                bias=self.bias_data,
                num_lanes=self.num_lanes,
                dw_in=16,
                dw_out=16,
                dw_bias=16,
                dtype="BF16",
                mif_init=False,
            )
        )

        self.auto_connect_weights()

        self.connect(self.input, self.dot.inputs.din)
        self.connect(self.dot.outputs.dout, self.bias.inputs.din)
        self.connect(self.bias.outputs.dout, self.output)


classic = Classic_weight_dag(
    weights=np.random.randn(64, 128).astype(np.float32),
    bias=np.random.randn(128).astype(np.float32),
)
classic(din=np.random.randn(1, 1, 64).astype(np.float32))
classic_dir = WORK / "classic_dump"
run_flow(classic, output_path=classic_dir)
lines = (classic_dir / "dump_files" / "weights.txt").open().readlines()[:5]
print("Bits per line: ", len(lines[0])-1) # -1 for newline
print("Preview classic weight txt:\n" + "".join(lines))


Dumping input and output data...
Dumping weights: matrix size: [64,128] total weights: 4096
Dumping bias weights. Total weights: 128
Running model CLASSIC_WEIGHT_DAG
Bits per line:  64
Preview classic weight txt:
0011111110100010101111101000010100111111001011011011111010010110
0011111110010001101111111011011000111111100010000011111100011011
0011111110100011001111011110111110111111011100101011111111100100
0011111101111101101111011101011100111110101011001011111101011010
1100000000110010001111110010100000111111001111110011111011011011

